# Protein Surprisal Atlas — Proteome Annotations + Associations (clean)

Annotates the scored proteome (~20k) and runs the associations. **CPU-only** (no GPU/torch). Reads the
scored chunks and writes outputs to your Google Drive (`MyDrive/psa_proteome`).

**Prerequisites**
- The proteome scores exist on Drive (from the scoring notebook).
- The two DepMap files are on Drive at `MyDrive/psa_proteome/data/external/`:
  `CRISPRGeneEffect.csv` and `CRISPRInferredCommonEssentials.csv`
  (figshare blocks Colab's IP, so these are downloaded manually in your own browser).

**Resumable.** Family-size (UniRef) + disorder (MobiDB) are cached on Drive; if the runtime resets,
just Run All again — it rebuilds and continues from the caches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST="/content/drive/MyDrive/psa_proteome"
assert os.path.isdir(PERSIST), f"{PERSIST} not found"
print("Persistent store:", PERSIST)

## 1. Clone repo + install deps (CPU; no torch)

In [ ]:
REPO="/content/protein-surprisal-atlas"
if os.path.isdir(REPO):
    !cd {REPO} && git fetch --quiet origin main && git checkout --quiet main && git pull --quiet
else:
    !git clone --branch main https://github.com/glenritschel/protein-surprisal-atlas.git {REPO}
!pip install -q pandas pyarrow statsmodels scipy scikit-learn requests tqdm matplotlib seaborn pyyaml biopython
!cd {REPO} && git log --oneline -1

## 2. Point data/ and results/ at Drive

In [ ]:
for name in ["data","results"]:
    !rm -rf {REPO}/{name}
    !ln -s {PERSIST}/{name} {REPO}/{name}
for sub in ["data/external","results/tables","results/reports","results/figures","results/logs"]:
    os.makedirs(f"{PERSIST}/{sub}", exist_ok=True)
import glob
n = len(glob.glob(f"{REPO}/results/tables/proteome_protein_scores_sampled_mask/part-*.parquet"))
print("proteome score chunk files:", n)
assert n > 0, "No proteome scores on Drive — run the scoring notebook first."


## 3. Verify the DepMap files are in place

In [ ]:
ext="/content/protein-surprisal-atlas/data/external"
depmap_ok=True
for w in ["CRISPRGeneEffect.csv","CRISPRInferredCommonEssentials.csv"]:
    p=os.path.join(ext,w)
    ok=os.path.exists(p) and os.path.getsize(p)>1000
    depmap_ok = depmap_ok and ok
    print(f"{w:36s} -> {'OK '+str(os.path.getsize(p)//1024//1024)+' MB' if ok else 'MISSING'}")
if not depmap_ok:
    print("\nDepMap file(s) missing. Download in your OWN browser and place on Drive at")
    print(f"  {PERSIST}/data/external/  with the exact names above:")
    print("   https://ndownloader.figshare.com/files/51064667  -> CRISPRGeneEffect.csv")
    print("   https://ndownloader.figshare.com/files/51064916  -> CRISPRInferredCommonEssentials.csv")
    print("Then re-run this cell. (You CAN proceed without them; DepMap will just be skipped.)")


## 4. Integrate annotations across the proteome
Family-size + disorder come from the Drive caches (instant if already fetched); gnomAD + DepMap are
read from `data/external`; low-complexity is computed locally. Writes `proteome_annotated.parquet`.
Long only if the caches aren't populated yet — resumable, so re-run on disconnect.

In [ ]:
!cd /content/protein-surprisal-atlas && PYTHONPATH=/content/protein-surprisal-atlas python scripts/integrate_annotations.py --scope proteome

## 5. Coverage across all annotations

In [ ]:
import pandas as pd
ann=pd.read_parquet("/content/protein-surprisal-atlas/results/tables/proteome_annotated.parquet")
n=len(ann); print(f"n = {n}\n")
for c in ["low_complexity_fraction","disorder_fraction","log_family_size",
          "gnomad_loeuf","gnomad_pli","depmap_gene_effect"]:
    if c in ann.columns:
        k=ann[c].notna().sum(); print(f"  {c:24s}: {k:6d}  ({100*k/n:.1f}%)")
    else:
        print(f"  {c:24s}: (not present)")
rep="/content/protein-surprisal-atlas/results/reports/proteome_annotation_coverage.md"
if os.path.exists(rep): print("\n"+open(rep).read())

## 6. Run the proteome-wide associations

In [ ]:
!cd /content/protein-surprisal-atlas && PYTHONPATH=/content/protein-surprisal-atlas python scripts/run_associations.py --scope proteome
import pandas as pd
t="/content/protein-surprisal-atlas/results/tables/proteome_association_results.csv"
print("\n=== proteome_association_results.csv ==="); display(pd.read_csv(t))
r="/content/protein-surprisal-atlas/results/reports/proteome_associations.md"
if os.path.exists(r): print(open(r).read())

## 7. View proteome association figures

In [ ]:
import glob,os
from IPython.display import Image, display
for p in sorted(glob.glob("/content/protein-surprisal-atlas/results/figures/proteome_fig*.png")):
    print(os.path.basename(p)); display(Image(filename=p, width=680))

## 8. Download the small outputs (reports + csv + proteome figures)

In [ ]:
import shutil,glob,os
stage="/content/proteome_annot_out"
for s in ["figures","reports","tables"]: os.makedirs(stage+"/"+s, exist_ok=True)
for p in glob.glob("/content/protein-surprisal-atlas/results/figures/proteome_fig*"): shutil.copy(p, stage+"/figures/")
for p in glob.glob("/content/protein-surprisal-atlas/results/reports/proteome_*"):     shutil.copy(p, stage+"/reports/")
for p in glob.glob("/content/protein-surprisal-atlas/results/tables/proteome_association_results.csv"): shutil.copy(p, stage+"/tables/")
shutil.make_archive("/content/proteome_annot_out","zip",stage)
from google.colab import files
print("Zipped:", os.path.getsize("/content/proteome_annot_out.zip")//1024, "KB")
files.download("/content/proteome_annot_out.zip")